In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn
from tqdm import tqdm

from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook


font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

output_notebook()
hv.extension('bokeh')


Loading BokehJS ...

In [2]:
monkey = 'fiona' # 'yasmin'  or 'fiona' 

# Load the pickle file
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'
cell_df = pd.read_pickle(pickle_file)
print(f"Unified DataFrame loaded from: {pickle_file}")
print(f"DataFrame shape: {cell_df.shape}")
print(cell_df.info())
cell_df.head()

Unified DataFrame loaded from: /home/barak/Projects/population_analysis/data/unified_cell_trial_data/unified_fiona_cell_trial_data.pkl
DataFrame shape: (1265818, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1265818 entries, 0 to 1265817
Data columns (total 27 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   cell_ID                 1265818 non-null  int64  
 1   cell_type               1265818 non-null  object 
 2   maestro_ID              1265818 non-null  int64  
 3   problem                 2741 non-null     object 
 4   grade                   1265818 non-null  int64  
 5   filename                1265818 non-null  object 
 6   trial_name              1265818 non-null  object 
 7   reaction_time           1197040 non-null  float64
 8   go_cue                  1265818 non-null  int64  
 9   stop_cue                563981 non-null   float64
 10  trial_failed            1265818 non-null  bool   
 11  s

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9867,msn,2,NaN,9,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,[1978.93],fi210824,a,255,fi210824a
1,9868,msn,3,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[40.85, 1162.37, 1952.2199999999998, 2059.2999...",fi210824,a,255,fi210824a
2,9869,msn,4,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,[],fi210824,a,255,fi210824a
3,9870,msn,5,NaN,9,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,[],fi210824,a,255,fi210824a
4,9871,msn,6,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[206.02, 257.5, 521.7, 611.32, 773.57, 818.050...",fi210824,a,255,fi210824a


In [3]:
cell_df[cell_df['cell_ID'] == 2049]['neural_data'].value_counts()

neural_data
[]    659
Name: count, dtype: int64

In [4]:
cell_df.columns

Index(['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename',
       'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed',
       'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade',
       'segs_durations', 'segs_times', 'trial_length', 'screen_rotation',
       'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session',
       'trial_number', 'trial_session'],
      dtype='object')

In [5]:
cell_df = cell_df[cell_df['grade'] <= 8].copy().reset_index(drop=True)
cell_df

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9868,msn,3,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[40.85, 1162.37, 1952.2199999999998, 2059.2999...",fi210824,a,255,fi210824a
1,9869,msn,4,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,[],fi210824,a,255,fi210824a
2,9871,msn,6,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[206.02, 257.5, 521.7, 611.32, 773.57, 818.050...",fi210824,a,255,fi210824a
3,9872,msn,7,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[641.98, 1061.1, 1297.57, 1795.7199999999998]",fi210824,a,255,fi210824a
4,9873,msn,8,NaN,7,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[79.0, 109.45, 162.12, 170.95000000000002, 235...",fi210824,a,255,fi210824a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984755,2924,pu msn,93,NaN,8,fi211020a.0629,STOP_L_SSD4,156.0,939,1167.0,...,1867,0.0,"[[0, 40], [7, 63], [1095, 1168]]",None,180,[],fi211020,a,629,fi211020a
984756,2927,pu msn,96,NaN,8,fi211020a.0629,STOP_L_SSD4,156.0,939,1167.0,...,1867,0.0,"[[0, 40], [7, 63], [1095, 1168]]",None,180,[],fi211020,a,629,fi211020a
984757,2929,pu msn,98,NaN,8,fi211020a.0629,STOP_L_SSD4,156.0,939,1167.0,...,1867,0.0,"[[0, 40], [7, 63], [1095, 1168]]",None,180,[],fi211020,a,629,fi211020a
984758,2934,pu msn,103,NaN,8,fi211020a.0629,STOP_L_SSD4,156.0,939,1167.0,...,1867,0.0,"[[0, 40], [7, 63], [1095, 1168]]",None,180,[],fi211020,a,629,fi211020a


In [6]:
cell_df['session'].nunique()


76

In [7]:
## Cell 3: Trial Type Distribution and Success Rates

# Create summary statistics for plotting
trial_summary = cell_df.groupby(['type', 'trial_failed']).size().reset_index(name='count')
trial_summary['outcome'] = trial_summary['trial_failed'].map({False: 'Success', True: 'Failed'})

# Calculate success rates by trial type
success_rates = cell_df.groupby('type').agg({
    'trial_failed': ['count', 'sum', 'mean']
}).round(3)
success_rates.columns = ['total_trials', 'failed_trials', 'failure_rate']
success_rates['success_rate'] = (1 - success_rates['failure_rate']) * 100
success_rates['failure_rate'] *= 100
print("Success rates by trial type:")
print(success_rates)

# Create the main visualization
plot1 = trial_summary.hvplot.bar(
    x='type', y='count', by='outcome',
    stacked=True,
    title=f'{monkey.title()} - Trial Distribution by Type and Outcome',
    xlabel='Trial Type',
    ylabel='Number of Trials',
    width=600, height=400,
    color=['#2E8B57', '#CD5C5C'],  # Green for success, red for failed
    legend='top_right'
)

plot1.opts(
    fontsize={'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12},
)
# success_rates

Success rates by trial type:
      total_trials  failed_trials  failure_rate  success_rate
type                                                         
CONT        228167          35434          15.5          84.5
GO          546140          20028           3.7          96.3
STOP        210453          94483          44.9          55.1


:Bars   [type,outcome]   (count)

In [8]:
## Cell 4: Success Rates by Trial Type (Percentage View)
# Create percentage view of success rates
trial_pct = cell_df.groupby('type').apply(
    lambda x: pd.Series({
        'Success': (1 - x['trial_failed'].mean()) * 100,
        'Failed': x['trial_failed'].mean() * 100
    })
).reset_index()

trial_pct_melted = trial_pct.melt(id_vars='type', var_name='outcome', value_name='percentage')

plot2 = trial_pct_melted.hvplot.bar(
    x='type', y='percentage', by='outcome',
    stacked=True,
    title=f'{monkey.title()} - Success Rate by Trial Type (%)',
    xlabel='Trial Type',
    ylabel='Percentage of Trials',
    width=600, height=400,
    color=['#2E8B57', '#CD5C5C'],
    legend='top',
    ylim=(0, 100)
)

plot2
# trial_pct
# trial_pct_melted

/tmp/ipykernel_41058/925790005.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trial_pct = cell_df.groupby('type').apply(


:Bars   [type,outcome]   (percentage)

In [9]:
from bokeh.palettes import Colorblind

# Histogram of cells per grade
print("=== CELLS PER GRADE ANALYSIS ===")

# Get the grade distribution for all cells
grade_counts = cell_df['grade'].value_counts().sort_index()
print(f"Grade distribution:")
for grade, count in grade_counts.items():
    print(f"  Grade {grade}: {count:,} cells")

print(f"\nTotal cells: {len(cell_df):,}")
print(f"Grade range: {cell_df['grade'].min()} - {cell_df['grade'].max()}")
print(f"Mean grade: {cell_df['grade'].mean():.2f}")
print(f"Median grade: {cell_df['grade'].median():.1f}")

# Create bar plot using hvplot with different colors per bar
grade_counts_df = cell_df.groupby('grade').size().reset_index(name='count')

# Create individual bars with different colors
bars = []
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
n_bars = len(grade_counts_df)
palette_size = min(8, max(3, n_bars))
colors = Colorblind[palette_size]

for i, (grade, count) in enumerate(zip(grade_counts_df['grade'], grade_counts_df['count'])):
    bar = hv.Bars([(grade, count)], kdims='grade', vdims='count').opts(
        color=colors[i], 
        alpha=0.8,
        width=10,
    )
    bars.append(bar)

# Overlay all bars
grade_bar = hv.Overlay(bars).opts(
    title=f'{monkey.title()} - Distribution of Cells per Grade',
    xlabel='Grade',
    ylabel='Number of Cells',
    width=700, height=400,
    fontsize=font_dict,
    xticks=list(grade_counts_df['grade'])
)

grade_bar

=== CELLS PER GRADE ANALYSIS ===
Grade distribution:
  Grade 6: 40,305 cells
  Grade 7: 208,163 cells
  Grade 8: 736,292 cells

Total cells: 984,760
Grade range: 6 - 8
Mean grade: 7.71
Median grade: 8.0


:Overlay
   .Bars.I   :Bars   [grade]   (count)
   .Bars.II  :Bars   [grade]   (count)
   .Bars.III :Bars   [grade]   (count)

In [10]:
# Violin plot of cell distribution by trial type and outcome
print("=== CELL DISTRIBUTION BY TRIAL TYPE AND OUTCOME ===")

# Count cells per trial for all trials - using correct column names
cells_per_trial = cell_df.groupby(['filename', 'type', 'trial_failed']).size().reset_index(name='cell_count')

# # Add outcome labels
cells_per_trial['outcome'] = cells_per_trial['trial_failed'].map({False: 'Success', True: 'Failed'})
cells_per_trial
print(f"Total trials analyzed: {len(cells_per_trial):,}")

print(f"\nCells per trial statistics by type and outcome:")
for trial_type in cells_per_trial['type'].unique():
    for outcome in ['Success', 'Failed']:
        type_outcome_data = cells_per_trial[
            (cells_per_trial['type'] == trial_type) & 
            (cells_per_trial['outcome'] == outcome)
        ]['cell_count']
        if len(type_outcome_data) > 0:
            print(f"  {trial_type} {outcome}: Mean={type_outcome_data.mean():.1f}, "
                  f"Median={type_outcome_data.median():.1f}, Min={type_outcome_data.min()}, "
                  f"Max={type_outcome_data.max()}, Trials={len(type_outcome_data)}")

# Create rotated violin plot with split by outcome using hv.Violin
violin_plot = hv.Violin(
    cells_per_trial, kdims=['type', 'outcome'], vdims='cell_count'
).opts(
    opts.Violin(
        show_legend=True, height=500, width=800,
        violin_color=hv.dim('outcome').str(),
        legend_position='top_right',
        split='outcome',
        title=f'{monkey.title()} - Distribution of Cells per Trial by Type and Outcome',
        xlabel='Trial Type',
        ylabel='Number of Cells per Trial',
        show_grid=True,
        violin_width=2,
        invert_axes=True,  # Keep normal orientation (vertical violins)
        tools=['hover'],
        fontsize=font_dict
    )
)


violin_plot

=== CELL DISTRIBUTION BY TRIAL TYPE AND OUTCOME ===
Total trials analyzed: 44,261

Cells per trial statistics by type and outcome:
  GO Success: Mean=22.3, Median=18.0, Min=1, Max=75, Trials=23572
  GO Failed: Mean=19.8, Median=16.0, Min=1, Max=70, Trials=1010
  STOP Success: Mean=21.2, Median=16.0, Min=1, Max=75, Trials=5479
  STOP Failed: Mean=23.4, Median=20.0, Min=1, Max=75, Trials=4040
  CONT Success: Mean=22.6, Median=19.0, Min=1, Max=75, Trials=8525
  CONT Failed: Mean=21.7, Median=17.0, Min=1, Max=75, Trials=1635


:Violin   [type,outcome]   (cell_count)

In [11]:
cell_df['ssd_len'].describe()

count    984760.000000
mean        313.993133
std         163.590369
min          24.000000
25%         168.000000
50%         450.000000
75%         450.000000
max         550.000000
Name: ssd_len, dtype: float64

In [12]:
# Bar plot of cell count by cell type and trial type
print("=== CELL TYPE DISTRIBUTION BY TRIAL TYPE ===")

# Get the cell type distribution by trial type
cell_type_trial_counts = cell_df.groupby(['type', 'cell_type']).size().reset_index(name='count')

# Calculate percentages within each trial type
trial_totals = cell_df.groupby('type').size()
cell_type_trial_counts['percentage'] = cell_type_trial_counts.apply(
    lambda row: (row['count'] / trial_totals[row['type']]) * 100, axis=1
)

print(f"Cell type distribution by trial type:")
for trial_type in cell_df['type'].unique():
    print(f"\n{trial_type} trials:")
    trial_data = cell_type_trial_counts[cell_type_trial_counts['type'] == trial_type].sort_values('count', ascending=False)
    for _, row in trial_data.iterrows():
        print(f"  {row['cell_type']}: {row['count']:,} cells ({row['percentage']:.1f}%)")

print(f"\nTotal cells: {len(cell_df):,}")
print(f"Trial types: {cell_df['type'].unique()}")
print(f"Unique cell types: {cell_df['cell_type'].nunique()}")

# Create grouped bar plot with proper separation and legend
# Use hvplot with explicit handling for the legend
cell_type_bar = cell_type_trial_counts.hvplot.bar(
    x='cell_type', y='count', by='type',
    title=f'{monkey.title()} - Distribution of Cells by Cell Type and Trial Type',
    xlabel='Cell Type',
    ylabel='Number of Cells',
    width=1000, height=600,
    alpha=0.8,
    rot=90,
    color=['#2E8B57', '#FF8C00', '#4169E1'],  # Green for GO, Orange for CONT, Blue for STOP
    legend='top_right'
)

# Apply additional styling options
cell_type_bar = cell_type_bar.opts(
    fontsize=font_dict,
    show_legend=True,
    legend_position='top_right',
    legend_opts={'click_policy': 'hide'}
)

cell_type_bar

=== CELL TYPE DISTRIBUTION BY TRIAL TYPE ===
Cell type distribution by trial type:

GO trials:
  pu msn: 270,444 cells (49.5%)
  msn: 156,494 cells (28.7%)
  hfdp: 56,111 cells (10.3%)
  tan: 13,360 cells (2.4%)
  lfd: 13,049 cells (2.4%)
  pu tan: 12,040 cells (2.2%)
  gpi: 7,212 cells (1.3%)
  unknown: 4,560 cells (0.8%)
  lfdb: 3,699 cells (0.7%)
  fiber: 3,025 cells (0.6%)
  fsn: 2,706 cells (0.5%)
  fef: 2,375 cells (0.4%)
  bd: 680 cells (0.1%)
  tan : 309 cells (0.1%)
  ctx: 76 cells (0.0%)

CONT trials:
  pu msn: 114,244 cells (50.1%)
  msn: 64,755 cells (28.4%)
  hfdp: 22,981 cells (10.1%)
  tan: 5,396 cells (2.4%)
  lfd: 5,389 cells (2.4%)
  pu tan: 5,137 cells (2.3%)
  gpi: 2,899 cells (1.3%)
  unknown: 2,090 cells (0.9%)
  lfdb: 1,501 cells (0.7%)
  fiber: 1,260 cells (0.6%)
  fsn: 1,082 cells (0.5%)
  fef: 1,012 cells (0.4%)
  bd: 288 cells (0.1%)
  tan : 115 cells (0.1%)
  ctx: 18 cells (0.0%)

STOP trials:
  pu msn: 103,982 cells (49.4%)
  msn: 60,786 cells (28.9%)
  hfd

:Bars   [cell_type,type]   (count)

In [13]:
# How many succesful trials per cell type and trial type
print("=== SUCCESSFUL TRIALS PER CELL TYPE AND TRIAL TYPE ===")
# Filter for successful trials only
successful_trials = cell_df[cell_df['trial_failed'] == False]
cell_type_trial_counts = successful_trials.groupby(['type', 'cell_type']).size().reset_index(name='count')

cell_type_trial_counts

# Create grouped bar plot with proper separation and legend
# Use hvplot with explicit handling for the legend
successful_cell_type_bar = cell_type_trial_counts.hvplot.bar(
    x='cell_type', y='count', by='type',
    title=f'{monkey.title()} - Successful Trials by Cell Type and Trial Type',
    xlabel='Cell Type',
    ylabel='Number of Successful Trials',
    width=1000, height=600,
    alpha=0.8,
    rot=90,
    color=['#2E8B57', '#FF8C00', '#4169E1'],  # Green for GO, Orange for CONT, Blue for STOP
    legend='top_right'
)
# Apply additional styling options
successful_cell_type_bar = successful_cell_type_bar.opts(
    fontsize=font_dict,
    show_legend=True,
    legend_position='top_right',
    legend_opts={'click_policy': 'hide'}
)   
successful_cell_type_bar


=== SUCCESSFUL TRIALS PER CELL TYPE AND TRIAL TYPE ===


:Bars   [cell_type,type]   (count)

In [14]:
msn_df = cell_df[cell_df['cell_type'].isin(['msn', 'pu msn'])].copy()
msn_df.attrs['max_grade'] = 8
msn_df.attrs['description'] = "DataFrame filtered to include only MSN and PU MSN cell types with a minimum grade of 8."
msn_df.attrs['update_stop_trial_failures_by_saccade_amplitude'] = True
msn_df.attrs['monkey'] = monkey

msn_df.to_pickle(save_path / f'msn_{monkey}_cell_trial_data.pkl')


In [17]:
msn_df

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9868,msn,3,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[40.85, 1162.37, 1952.2199999999998, 2059.2999...",fi210824,a,255,fi210824a
1,9869,msn,4,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,[],fi210824,a,255,fi210824a
2,9871,msn,6,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[206.02, 257.5, 521.7, 611.32, 773.57, 818.050...",fi210824,a,255,fi210824a
3,9872,msn,7,NaN,8,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[641.98, 1061.1, 1297.57, 1795.7199999999998]",fi210824,a,255,fi210824a
4,9873,msn,8,NaN,7,fi210824a.0255,GO_R,263.0,1035,NaN,...,2186,0.0,"[[157, 229], [1298, 1368], [1344, 1383]]",None,0,"[79.0, 109.45, 162.12, 170.95000000000002, 235...",fi210824,a,255,fi210824a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984755,2924,pu msn,93,NaN,8,fi211020a.0629,STOP_L_SSD4,156.0,939,1167.0,...,1867,0.0,"[[0, 40], [7, 63], [1095, 1168]]",None,180,[],fi211020,a,629,fi211020a
984756,2927,pu msn,96,NaN,8,fi211020a.0629,STOP_L_SSD4,156.0,939,1167.0,...,1867,0.0,"[[0, 40], [7, 63], [1095, 1168]]",None,180,[],fi211020,a,629,fi211020a
984757,2929,pu msn,98,NaN,8,fi211020a.0629,STOP_L_SSD4,156.0,939,1167.0,...,1867,0.0,"[[0, 40], [7, 63], [1095, 1168]]",None,180,[],fi211020,a,629,fi211020a
984758,2934,pu msn,103,NaN,8,fi211020a.0629,STOP_L_SSD4,156.0,939,1167.0,...,1867,0.0,"[[0, 40], [7, 63], [1095, 1168]]",None,180,[],fi211020,a,629,fi211020a


In [15]:
msn_df['screen_rotation'].value_counts()

screen_rotation
0.0    770705
Name: count, dtype: int64